# Privacy and Governance Analysis

This notebook has the intention to analyse the risks regarding privacy on the start-ups credit aplications dataset. Meaning that it evaluates the privacy, regulatory and governance implications of the findings indentified in the previous project analyses.  

NovaCred is a fintech company using machine learning models to support credit approval decisions. Because credit scoring directly affects individuals' access to financial services, such systems involve sensitive personal data and significant regulatory obligations.

This notebook examines whether the dataset and analytical pipeline meet key requirements related to:
- personal data protection  
- data governance  
- regulatory compliance  
- responsible AI practices


### Relationship with Previous Analyses

This notebook builds directly on the findings of Notebook 02 (Bias Analysis) and addresses the privacy and governance obligations that those findings trigger. Credit scoring is classified as a high-risk AI system under the EU AI Act (Annex III, No. 5b), which means it is subject to both GDPR and AI Act requirements simultaneously.

Examples include:
- duplicate SSNs linked to multiple individuals  
- demographic attributes used in bias analysis  
- proxy variables that may reveal protected characteristics  
- missing governance metadata such as consent timestamps or audit logs

This analysis was performed on the dataset that was already cleaned and prepared by the data team. It will be possile to focus on privacy and governance, preventing techinical mistakes related to data quality.

## 0. Setup and Load Data

### Sections
0. Setup & Load Data
1. Identification of Personal Data & Classification 
2. Privacy Risk Mitigation
3. Pseudonymization
4. GDPR Compliance Mapping 
5. Governance Gaps
6. Governance Recommendations
7. Conclusion


---

### Regulatory Framework

| Regulation | Relevance |
|---|---|
| **GDPR** (Regulation (EU) 2016/679) | Governs all processing of personal data — lawful basis, data minimisation, storage limitation, rights of data subjects |
| **EU AI Act** (Regulation (EU) 2024/1689) | Credit scoring = high-risk AI (Annex III) — requires risk management, data governance, transparency, human oversight |

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import hashlib

df = pd.read_csv('clean_dataset_view.csv')

In [2]:
df.columns
columns_table = df.columns.to_frame(index=False, name='Columns')
columns_table

,Columns
0,_id
1,spending_behavior
2,processing_timestamp
3,full_name
4,email
5,ssn
6,ip_address
7,gender
8,date_of_birth
9,zip_code


## 1. Identification of Personal Data & Classification

The first step in the privacy and governance assessment is to identify which attributes in the dataset cnstitute personal data. 

Under GDPR Art. 4(1), personal data refers to information that allows the identification of an individual, either directly (e.g. name or SSN) or indirectly through the combination of multiple attrributes. For that reason, it should be treated with special security and privacy measures. In this dataset, several potential fields were identified as Personally Identifiable Information (PII). 

Based on this framework, we classify dataset attributes according to their potential identifiability risk.

After analysing the columns of the dataset, it is possible to conclude that the following columns are considered personal data. We classify each field by PII type and risk level.
- **Direct identifiers** are attributes that can uniquely identify an individual on their own, such as name, email or SSN.
- **Quasi-identifiers** do not identify a person individually but may allow identification when combined with other attributes, such as date of birth, ZIP code or gender.
- **Technical identifier** contains information that might identify an individual or its devices.
- **Non-personal / Derived attributes** are variables representing system outputs or aggregated results that do not direcrly identify individuals.

| Category | Definition | Examples in Dataset |
|--------|-------------|--------------------|
| **Direct Identifier** | Uniquely identifies a person on its own | full_name, ssn, email |
| **Quasi-Identifier (QI)** | Combination with other attributes may enable identification | date_of_birth, zip_code, gender |
| **Technical Identifier** | Identifies a device or interaction | ip_address |
| **Non-Personal / Derived** | Represents system outputs rather than personal identity | loan_approved, interest_rate |

Although the attributes such as Zip code are not considered sensitive categories under GDPR Art.9, they may act as proxy variables for protected characteristics such as race or socio economic status.
When such attributes are used in automated decision-making systems,they may contribute to indirect discrimination, which was explored in the bias analysis performed in Notebook 02.

This highlights the importance of assessing both privacy risk and algorithmic fairness risks in the governance framework. 

In [ ]:
pii_classification = [
    {
        'Column':        '_id',
        'PII Category':  'Direct Identifier',
        'Risk' :        '🔴 High',
        'Justification': 'Primary key linking directly to the record; enables re-identification when combined with other fields.',
        'Recommended Technique': 'Pseudonymisation.',
        'Bias NB Link':  '—'
    },
    {
        'Column':        'full_name',
        'PII Category':  'Direct Identifier',
        'Risk' :        '🔴 High',
        'Justification': 'Direct identifier of the applicant; duplicate name–SSN findings increase re-identification risk.',
        'Recommended Technique': 'Pseudonymisation via hash (HMAC-SHA256 + salt)',
        'Bias NB Link':  'Duplicate SSN / name mismatch flagged'
    },
    {
        'Column':        'ssn',
        'PII Category':  'Direct Identifier (Sensitive)',   
            'Risk' :        '🔴 High',
        'Justification': 'Highly sensitive national identifier; duplicate values indicate integrity and breach risk.',
        'Recommended Technique': 'Tokenisation.',
        'Bias NB Link':  'Data quality NB: duplicate SSN finding'
    },
    {
        'Column':        'date_of_birth',
        'PII Category':  ' Quasi-Identifier (High Risk)',
        'Risk' :        '🟠 High',
        'Justification': 'Used only to derive age groups; storing the full birth date exceeds data minimisation needs.',
        'Recommended Technique': 'Generalisation',
        'Bias NB Link':  'Section 3 — Age-Based Discrimination; Section 5 — Interaction Effects'
    },
    {
        'Column':        'gender',
        'PII Category':  'Quasi-Identifier (Protected Attribute)',
        'Risk' :        '🟠 High',
        'Justification': 'Protected attribute used for fairness auditing; disparate impact detected (DI = 0.77).',
        'Recommended Technique': 'Governance Control',
        'Bias NB Link':  'Section 2 — Gender Disparate Impact (DI = 0.77)'
    },
    {
        'Column':        'zip_code',
        'PII Category':  'Quasi-Identifier (Proxy Risk)',
         'Risk' :        '🟠 High'  ,
        'Justification': 'Strong demographic proxy; high correlation with gender indicates indirect bias risk.',
        'Recommended Technique': 'Geographic generalisation',
        'Bias NB Link':  'Section 4 — Proxy Variable Analysis (corr = −0.805)'
    },
    {
        'Column':        'annual_income',
        'PII Category':  'Quasi-Identifier (Moderate Risk)',
            'Risk' :        '🟡 Medium'     ,
        'Justification': 'Legitimate credit feature but moderately correlated with age.',
        'Recommended Technique': 'Differential privacy',
        'Bias NB Link':  'Section 4 — Proxy Variable Analysis (corr_age = 0.394)'
    },
    {
        'Column':        'credit_history_months',
        'PII Category':  'Quasi-Identifier (Age Proxy)',
            'Risk' :        '🟡 Medium'     ,
        'Justification': 'Strongest age proxy in dataset and key driver of disadvantage for younger applicants.',
        'Recommended Technique': 'Generalisation',
        'Bias NB Link':  'Section 4 — Primary age proxy (corr = 0.649)'
    },
    {
        'Column':        'savings_balance',
        'PII Category':  'Quasi-Identifier (Low-Medium Risk)',
            'Risk' :        '🟡 Medium'     ,
        'Justification': 'Financial feature with mild age correlation; moderate proxy risk.',
        'Recommended Technique': 'Generalisation',
        'Bias NB Link':  'Section 4 — Borderline proxy (corr_age = 0.286)'
    },
    {
        'Column':        'debt_to_income',
        'PII Category':  'Quasi-Identifier (Low Risk)',
            'Risk' :        '🟢 Low'     ,
        'Justification': 'Low proxy correlations; standard indicator of creditworthiness.',
        'Recommended Technique': 'None',
        'Bias NB Link':  'Notebook 01 — negative value truncation'
    },
    {
        'Column':        'spending_category',
        'PII Category':  'Quasi-Identifier (Behavioural)',
            'Risk' :        '🟠 High'     ,
        'Justification': 'Behavioural profiling feature and strongest predictor of approval outcome.',
        'Recommended Technique': 'Generalisation',
        'Bias NB Link':  'Section 4 — Strongest outcome predictor'
    },
    {
        'Column':        'spending_amount',
        'PII Category':  'Quasi-Identifier (Negligible Risk)',
            'Risk' :        '🟢 Low'     ,
        'Justification': 'No predictive value or demographic correlation; unnecessary for modelling.',
        'Recommended Technique': 'Deletion.',
        'Bias NB Link':  'Section 4 — No predictive value'
    },
    {
        'Column':        'loan_approved',
        'PII Category':  'Quasi-Identifier (Decision Output)',
                'Risk' :        '🟡 Medium'     ,
        'Justification': 'Automated decision outcome requiring auditability and explainability.',
        'Recommended Technique': 'Audit Logging',
        'Bias NB Link':  'Sections 2–5 — Bias metrics'
    },
    {
        'Column':        'processing_timestamp',
        'PII Category':  'Quasi-Identifier (Metadata)',
            'Risk' :        '🟢 Low'     ,
        'Justification': 'Links the record to a point in time and enables retention control.',
        'Recommended Technique': 'Retention Control',
        'Bias NB Link':  '—'
    },
]

pii_df = pd.DataFrame(pii_classification)


display(pii_df.set_index('Column')[['PII Category','Risk','Recommended Technique','Bias NB Link']])

,PII Category,Risk,Recommended Technique,Bias NB Link
Column,,,,
_id,Direct Identifier,🔴 High,Pseudonymisation.,—
full_name,Direct Identifier,🔴 High,Pseudonymisation via hash (HMAC-SHA256 + salt),Duplicate SSN / name mismatch flagged
ssn,Direct Identifier (Sensitive),🔴 High,Tokenisation.,Data quality NB: duplicate SSN finding
date_of_birth,Quasi-Identifier (High Risk),🟠 High,Generalisation,Section 3 — Age-Based Discrimination; Section ...
gender,Quasi-Identifier (Protected Attribute),🟠 High,Governance Control,Section 2 — Gender Disparate Impact (DI = 0.77)
zip_code,Quasi-Identifier (Proxy Risk),🟠 High,Geographic generalisation,Section 4 — Proxy Variable Analysis (corr = −0...
annual_income,Quasi-Identifier (Moderate Risk),🟡 Medium,Differential privacy,Section 4 — Proxy Variable Analysis (corr_age ...
credit_history_months,Quasi-Identifier (Age Proxy),🟡 Medium,Generalisation,Section 4 — Primary age proxy (corr = 0.649)
savings_balance,Quasi-Identifier (Low-Medium Risk),🟡 Medium,Generalisation,Section 4 — Borderline proxy (corr_age = 0.286)


The recommended techniques refer to standard privacy-preserving methods such as pseudonymisation, tokenisation, generalisation, and differential privacy, which are widely recommended in GDPR-compliant data governance frameworks.

#### Legal Basis Summary
Based on the legal and conceptual framework described above, each dataset attribute was evaluated according to its identifiability risk and potential impact on privacy and algorithmic fairness. Direct identifiers such as full_name and ssn fall clearly within the definition of personal data under GDPR Art. 4(1). Quasi-identifiers including date_of_birth, zip_code, and financial attributes may not identify individuals alone but can enable re-identification when combined with other variables. Additionally, behavioural and decision-related attributes (e.g., spending_category and loan_approved) are relevant for profiling and automated decision-making considerations under GDPR Art. 4(4) and Art. 22.


## 3. Pseudonymization

After identifying and classifying the personal data present in the dataset, the next step is to apply technical measures designed to reduce the risk of direct identification of individuals. One of the main techniques recommended by the GDPR for the protection of personal data is pseudonymization, which consists of replacing direct identifiers with artificial representations that do not allow a person to be identified without access to additional information.

In the context of this credit application dataset, pseudonymization is particularly relevant for attributes that constitute direct identifiers, such as name, email address, SSN, or IP address. These fields allow for the immediate identification of an individual and, therefore, represent a high risk in case of unauthorized access or misuse of the data. 

For this project, Pseudonymization is applied to direct identifiers that are not necessary for the analysis of the credit decision process. In this way, the exposure of sensitive data is reduced without compromising the analytical capacity of the dataset.

It is important to note that pseudonymization significantly reduces the risk of direct identification of individuals, but does not completely eliminate this risk. According to the GDPR, pseudonymized data continues to be considered personal data, since re-identification may be possible if there is access to additional information or through combination with other variables in the dataset. For this reason, pseudonymization should be seen as a risk mitigation measure and not as a complete anonymization solution. This distinction between pseudonymization and anonymization is recognized in the GDPR Recital 26.

In [6]:
def pseudonymize(value):
    if pd.isna(value):
        return None
    return hashlib.sha256(str(value).encode()).hexdigest()

pii_columns = ["full_name", "email", "ssn", "ip_address"]

for col in pii_columns:
    df[col + "_pseudo"] = df[col].apply(pseudonymize)

df = df.drop(columns=pii_columns)
df.head()

,_id,spending_behavior,processing_timestamp,gender,date_of_birth,zip_code,annual_income,credit_history_months,debt_to_income,savings_balance,loan_approved,rejection_reason,loan_purpose,interest_rate,approved_amount,notes,full_name_pseudo,email_pseudo,ssn_pseudo,ip_address_pseudo
0,app_001,"[{'category': 'Fitness', 'amount': 576}]",NaN,female,1986-05-27,90230,102000.0,37,0.42,0,False,high_dti_ratio,NaN,NaN,NaN,DUPLICATE_ENTRY_ERROR,574bb517d8a0854fc6906b690627290925f1e8d5cd66f9...,137c1b4a89343995539bffbc9ce6edaa65bcb498fc0434...,d49cad59b567ffb59422b169add8cf587764aabe1b91f6...,2ac8bc04be2fbb6540f7d976fda8091de37136368fa662...
1,app_002,"[{'category': 'Education', 'amount': 533}]",NaN,male,1999-08-01,10020,41000.0,5,0.36,18200,False,algorithm_risk_score,NaN,NaN,NaN,NaN,a6df505311f83600f5db73028630064a1243ff7f5874dd...,ad586ff31ede0ddcf626645f064f888e0f8e7beb3f81e2...,0e1892639ce70228710f0735fa85377403b32b612f398c...,a96122c4b0370a28986788dc0eb2030aa9c254a19736b0...
2,app_003,"[{'category': 'Healthcare', 'amount': 450}]",NaN,female,1982-08-24,90213,65000.0,74,0.43,7090,True,NaN,NaN,3.4,76000.0,NaN,d2802474636ba97c1cd42a7dde4faba876eade9b812ff6...,f110fede2c1f19f308c6cbf664f079f53438e00dfe44d6...,5e93b5041cec1a785d060c243cd2a8f8b80493574aebdd...,a6be52c95db995dc90b7718c92acd95a5a6aa1bb852f87...
3,app_004,"[{'category': 'Transportation', 'amount': 329}...",NaN,female,02/28/1995,90217,69000.0,9,0.41,10327,False,high_dti_ratio,NaN,NaN,NaN,NaN,1d462c9ea4cb62c4b2604e5b5d3ff78b403d5cc29b96cf...,a094d9056654b513ce0e2f5c254f33470604a4549c72d3...,3ef30b5e161efd64477f8094eafe1546f79df340fa9ad6...,e640905a96e360ac1b7a7535a37024dc1875e7d06aa417...
4,app_005,"[{'category': 'Insurance', 'amount': 585}]",NaN,female,1960/06/19,90296,39000.0,76,0.06,15011,False,algorithm_risk_score,NaN,NaN,NaN,NaN,14576b60592c0da19715475876d40dcaf0625b9a78b7df...,1a407bcae335e67007abb52c9b1766b7b9cd78403829ff...,8c2a4e7afef60f4e60cccf69d18a73b7ccb2e329904802...,416311facac9e50393e42001611b32a0d7b8af4c964fd6...


#### Retained attributes for fairness analysis
It is important to note that some attributes are not pseudonymized at this stage intentionally, as they are necessary to assess potential patterns of discrimination in credit decisions. Demographic and financial variables are retained to allow for fairness audits, and are subsequently analyzed for potential biases.

This approach seeks to balance privacy protection and analytical transparency, aligning with the fundamental principles of the GDPR, namely data minimization, security of processing, and accountability.

## 4. GDPR Compliance Mapping

After applying privacy protection measures such as pseudonymization, it is important to evaluate how the data processing practices observed in the dataset relate to the key principles established by the General Data Protection Regulation (GDPR). This section connects the findings of the analysis with relevant GDPR requirements, highlighting both areas of alignment and potential governance gaps.

In [ ]:
gdpr_compliance = pd.DataFrame({
    "Article": [
        "Article 6",
        "Article 5(1)(b)",
        "Article 5(1)(c)",
        "Article 5(1)(d)",
        "Article 5(1)(e)",
        "Article 32",
        "Article 22"
    ],

    "Key Principle": [
        "Lawfulness & Transparency",
        "Purpose Limitation",
        "Data Minimization",
        "Accuracy",
        "Storage Limitation",
        "Integrity & Confidentiality",
        "Human oversight in automated decisions"
    ],

    "Meaning": [
        "Personal data must be processed lawfully and transparently.",
        "Data must be collected for specific, explicit purposes.",
        "Only data necessary for the decision should be processed.",
        "Personal data must be accurate and up to date.",
        "Personal data should not be stored longer than necessary.",
        "Data must be protected against unauthorized access.",
        "Individuals have the right not to be subject solely to automated decisions."
    ],

    "Evidence in Dataset": [
        "The dataset contains no consent_timestamp field, meaning there is no evidence that users gave explicit consent for processing personal data.",
        "Behavioral variables such as spending_category and spending_amount are used for credit decisions, but the dataset does not document whether users were informed that this data would be used for credit approval.",
        "Sensitive attributes such as date_of_birth (used to derive age) and gender are included. Additionally, zip_code acts as a strong proxy for gender (corr ≈ -0.805), suggesting the model may indirectly use demographic information.",
        "Derived variables such as age are calculated from date_of_birth. Without validation or update mechanisms, these attributes may become inaccurate over time.",
        "No retention policy or timestamp fields are present to indicate how long personal data is stored, suggesting lack of storage limitation governance.",
        "The dataset contains personal financial information (income, debt_to_income, savings_balance), but there is no evidence of anonymization, encryption, or access control.",
        "Loan approval appears to be determined through automated scoring features. There is no indication of human review or override mechanisms for automated decisions."
    ],

    "Recommended Action": [
        "Implement explicit consent tracking by adding a consent_timestamp field and documenting the legal basis for processing.",
        "Document the exact purpose for each feature used in the credit decision model and ensure customers are informed about behavioral data usage.",
        "Remove unnecessary demographic attributes or replace them with less sensitive proxies. Evaluate whether zip_code should be removed or aggregated to reduce proxy bias.",
        "Implement data validation pipelines and ensure derived attributes such as age are recalculated dynamically rather than stored statically.",
        "Define and document a clear data retention policy and introduce retention timestamps for automatic deletion or anonymization.",
        "Implement encryption, role-based access control, and audit logs to protect sensitive financial data.",
        "Introduce human-in-the-loop review for borderline loan decisions and provide explainability and appeal mechanisms for automated outcomes."
    ]
})


Overall, the privacy measures applied support GDPR principles. However, the absence of clear governance mechanisms, such as defined data retention policies or documented data management procedures, indicates potential areas for improvement.

## 5. Governance Gaps & Recommendations
Beyond the privacy risks previously identified, it is important to analyse whether the dataset contains the necessary fields to guarantee compliance with Data Governance good practices and GDPR principles.

An essential part of data governance consists of ensuring that there is sufficient documentation to justify the collection, use, and retention of personal data. However, while analysing the structure of the dataset, we observed that some important fields needed to ensure transparency and compliance seem to be missing.

During the analysis of the dataset and the associated data processing practices, several potential governance gaps were identified. Although certain privacy protection measures were applied in this project, such as the pseudonymization of direct identifiers, the dataset still reveals areas where stronger data governance mechanisms could be implemented to reduce privacy risks and improve transparency in the credit decision process.

**1. Exposure of Direct Identifiers**  
The dataset contains several attributes that can directly identify individuals, including full_name, email, ssn, and ip_address. These identifiers represent highly sensitive personal data and may create significant privacy risks if they are stored or processed without appropriate protection mechanisms.  
Although pseudonymization was applied during this analysis to mitigate these risks, the presence of these identifiers in the original dataset highlights the importance of implementing stronger safeguards, such as encryption, access controls, and restricted data access policies directly in their operational data pipelines.

**2. Governance Behavioral Data**   
The dataset includes behavioral information related to applicants' spending patterns, stored in the spending_behavior field. While this information may support credit risk assessment, the use of behavioral data may also raise governance concerns related to profiling and proportionality.  
Nova's Credit should conduct data necessity assessments and clearly document the purpose of collecting behavioral data in order to comply with the data minimization principle (Article 5(1)(c) GDPR).

**3. Definition of Data Retention Policies**    
The organization should define clear data retention policies, specifying how long personal data can be stored and when it should be deleted.These policies should align with the storage limitation principle defined in Article 5(1)(e) of the GDPR, ensuring that personal data is not retained longer than necessary.  
Automated deletion or anonymization mechanisms can help enforce these retention policies in practice.

**4. Limitation of Access to Personal Data**    
Access to personal data should be restricted to authorized users only.
This can be done through access control mechanisms.

**5. Documentation of the Purpose of Processing**    
The system should include information about the purpose of collecting and using personal data, for example through a processing_purpose field.
These measures help reduce privacy risks, improve transparency, and ensure greater compliance with the principles of the GDPR

**6.Limited Transparency of Credict Decisions**  
The dataset includes the final outcome of credit decisions, such as whether a loan was approved or rejected. However, it does not provide information about the criteria, models, or decision rules used to generate these outcomes.  
This lack of transparency may represent a governance concern, particularly in the context of automated decision-making systems. According to Article 22 of the GDPR, individuals have rights related to decisions that significantly affect them when those decisions are based on automated processing.  
To address the lack of transparency in the credit decision process, organizations should implement mechanisms that improve the explainability of automated decisions. This may include documenting decision rules, providing model explanations, or maintaining audit trails that allow decisions to be reviewed and justified when required.Such measures help support transparency and accountability, particularly in relation to automated decision-making under Article 22 of the GDPR.

**7. Lack of Explicit Data Governance**   
Although the dataset includes a processing_timestamp attribute, it does not provide information about how long personal data should be retained or when it should be deleted. The absence of clearly defined retention rules may create risks of storing personal data longer than necessary.  
This issue relates to the storage limitation principle defined in Article 5(1)(e) of the GDPR, which requires organizations to ensure that personal data is not retained indefinitely without a justified purpose.

**8. Human Oversight in Automated Credit Decisions**  
Since credit scoring systems may involve automated decision-making that significantly affects individuals, it is important to ensure that appropriate human oversight mechanisms are in place. Human review processes can help detect potential errors, biases, or unfair outcomes produced by automated systems.  
This approach is consistent with the requirements for high-risk AI systems defined in the EU AI Act, which emphasize the importance of human oversight in AI-assisted decision-making.

In [8]:
governance_summary = pd.DataFrame({
    "Governance Gap": [
        "Exposure of direct identifiers",
        "Use of behavioral data",
        "Limited transparency of credit decisions",
        "Lack of retention governance",
        "Unrestricted access to personal data",
        "Lack of documented processing purpose",
        "Lack of human oversight in automated decisions"
    ],
    
    "Evidence in Dataset": [
        "full_name, email, ssn, ip_address",
        "spending_behavior",
        "loan_approved without explanation",
        "processing_timestamp without retention rules",
        "No access control information available",
        "No explicit field documenting processing purpose",
        "Automated credit decision outcome without review mechanism"
    ],
    
    "Potential Risk": [
        "Direct identification of individuals",
        "Profiling and potential bias",
        "Opaque automated decision-making",
        "Data stored longer than necessary",
        "Unauthorized access to personal data",
        "Lack of transparency in data processing",
        "Unfair or biased automated decisions without human review"
    ],
    
    "Recommended Action": [
        "Implement pseudonymization, encryption and access control",
        "Assess necessity of behavioral data collection",
        "Introduce explainability mechanisms and decision audit trails",
        "Define retention schedule and automated deletion policies",
        "Implement role-based access control policies",
        "Document purpose of personal data processing",
        "Introduce human review mechanisms for automated credit decisions"
    ],
    
    "Regulatory Reference": [
        "GDPR Art. 32 – Security of processing",
        "GDPR Art. 5(1)(c) – Data minimization",
        "GDPR Art. 22 – Automated decision-making",
        "GDPR Art. 5(1)(e) – Storage limitation",
        "GDPR Art. 32 – Security of processing",
        "GDPR Art. 5(1)(a) – Lawfulness, fairness, transparency",
        "EU AI Act – Human oversight for high-risk AI systems"
    ]
})

governance_summary

,Governance Gap,Evidence in Dataset,Potential Risk,Recommended Action,Regulatory Reference
0,Exposure of direct identifiers,"full_name, email, ssn, ip_address",Direct identification of individuals,"Implement pseudonymization, encryption and acc...",GDPR Art. 32 – Security of processing
1,Use of behavioral data,spending_behavior,Profiling and potential bias,Assess necessity of behavioral data collection,GDPR Art. 5(1)(c) – Data minimization
2,Limited transparency of credit decisions,loan_approved without explanation,Opaque automated decision-making,Introduce explainability mechanisms and decisi...,GDPR Art. 22 – Automated decision-making
3,Lack of retention governance,processing_timestamp without retention rules,Data stored longer than necessary,Define retention schedule and automated deleti...,GDPR Art. 5(1)(e) – Storage limitation
4,Unrestricted access to personal data,No access control information available,Unauthorized access to personal data,Implement role-based access control policies,GDPR Art. 32 – Security of processing
5,Lack of documented processing purpose,No explicit field documenting processing purpose,Lack of transparency in data processing,Document purpose of personal data processing,"GDPR Art. 5(1)(a) – Lawfulness, fairness, tran..."
6,Lack of human oversight in automated decisions,Automated credit decision outcome without revi...,Unfair or biased automated decisions without h...,Introduce human review mechanisms for automate...,EU AI Act – Human oversight for high-risk AI s...


These recommendations aim to strengthen data governance practices, reduce privacy risks, and support the responsible use of personal data in automated credit decision systems, in line with both GDPR requirements and emerging AI governance frameworks such as the EU AI Act.

## 7. Conclusion
In summary, this work allowed us to analyze the privacy and data usage risks associated with NovaCred's credit approval system. The analysis showed that the dataset contains several types of personal data and identifiers that could create risks of customer identification if not properly protected.

The results also indicate that some measures are already in place to mitigate these risks, such as the removal of direct identifiers and the application of pseudonymization techniques. However, several aspects can still be improved, including the minimization of certain data attributes, the definition of clear data retention policies, and greater transparency regarding how credit decisions are generated.

Overall, the implementation of stronger data governance and privacy protection practices would help reduce privacy risks and support greater compliance with regulations such as the GDPR. These measures would contribute to making the credit decision system more secure, transparent, and responsible.